# 🥇 Gold Layer - Customer 360 View
## AWS S3 External Storage via Unity Catalog

**Purpose**: Create comprehensive customer 360 view with all customer interactions

**Source**: `workspace.`workspace-adventureworks-silver`` (S3)
**Target**: `workspace.`workspace-adventureworks-gold`` (S3)

**Gold Tables to Create**:
1. **gold_customer_360** - Complete customer profile with metrics
2. **gold_customer_segmentation** - RFM-based customer segments
3. **gold_customer_lifetime_value** - Customer lifetime value analysis

**Business Metrics**:
- Recency, Frequency, Monetary (RFM) analysis
- Customer lifetime value (CLV)
- Customer acquisition and churn indicators
- Geographic distribution
- Purchase patterns and preferences

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql import Row
from datetime import datetime
import time

# Configuration
SILVER_SCHEMA = "workspace.`workspace-adventureworks-silver`"
GOLD_SCHEMA = "workspace.`workspace-adventureworks-gold`"

print("=" * 80)
print("🥇 GOLD LAYER - CUSTOMER 360 VIEW")
print("=" * 80)
print(f"Start Time: {datetime.now()}")
print(f"Source: {SILVER_SCHEMA} (AWS S3)")
print(f"Target: {GOLD_SCHEMA} (AWS S3)")
print("=" * 80)
print()

transformation_results = []

In [0]:
# Customer 360 View
print("\n" + "=" * 80)
print("👥 BUILDING gold_customer_360")
print("=" * 80)

start_time = time.time()

try:
    # Read from Silver
    dim_customer = spark.table(f"{SILVER_SCHEMA}.dim_customer").filter(col("is_current") == True)
    fact_sales_order = spark.table(f"{SILVER_SCHEMA}.fact_sales_order")
    fact_sales_detail = spark.table(f"{SILVER_SCHEMA}.fact_sales_order_detail")
    dim_date = spark.table(f"{SILVER_SCHEMA}.dim_date")
    dim_location = spark.table(f"{SILVER_SCHEMA}.dim_location").filter(col("is_current") == True)
    
    # Calculate customer metrics
    customer_orders = (fact_sales_order
        .join(dim_date, fact_sales_order.order_date_sk == dim_date.date_sk)
        .groupBy(col("customer_sk"))
        .agg(
            count("sales_order_id").alias("total_orders"),
            sum("total_due_amount").alias("total_revenue"),
            avg("total_due_amount").alias("avg_order_value"),
            min("date_value").alias("first_order_date"),
            max("date_value").alias("last_order_date"),
            sum(when(col("is_online_order") == True, 1).otherwise(0)).alias("online_order_count"),
            sum(when(col("is_online_order") == False, 1).otherwise(0)).alias("offline_order_count")
        )
    )
    
    # Calculate product diversity
    customer_products = (fact_sales_detail
        .groupBy(col("customer_sk"))
        .agg(
            countDistinct("product_sk").alias("unique_products_purchased"),
            sum("order_quantity").alias("total_units_purchased")
        )
    )
    
    # Get most recent shipping location
    recent_location = (fact_sales_order
        .join(dim_date, fact_sales_order.order_date_sk == dim_date.date_sk)
        .withColumn(
            "row_num",
            row_number().over(Window.partitionBy("customer_sk").orderBy(col("date_value").desc()))
        )
        .filter(col("row_num") == 1)
        .select(
            col("customer_sk"),
            col("ship_to_location_sk").alias("primary_shipping_location_sk")
        )
    )
    
    # Build Customer 360
    gold_customer_360 = (dim_customer
        .join(customer_orders, dim_customer.dim_customer_sk == customer_orders.customer_sk, "left")
        .join(customer_products, dim_customer.dim_customer_sk == customer_products.customer_sk, "left")
        .join(recent_location, dim_customer.dim_customer_sk == recent_location.customer_sk, "left")
        .join(
            dim_location.select(
                col("dim_location_sk"),
                col("city").alias("shipping_city"),
                col("state_province").alias("shipping_state"),
                col("country_region").alias("shipping_country")
            ),
            col("primary_shipping_location_sk") == col("dim_location_sk"),
            "left"
        )
        .select(
            col("customer_id"),
            col("first_name"),
            col("last_name"),
            concat_ws(" ", col("first_name"), col("last_name")).alias("full_name"),
            col("company_name"),
            col("email_address"),
            col("phone"),
            col("shipping_city"),
            col("shipping_state"),
            col("shipping_country"),
            coalesce(col("total_orders"), lit(0)).alias("lifetime_orders"),
            coalesce(col("total_revenue"), lit(0)).alias("lifetime_revenue"),
            col("avg_order_value"),
            coalesce(col("unique_products_purchased"), lit(0)).alias("unique_products"),
            coalesce(col("total_units_purchased"), lit(0)).alias("total_units"),
            coalesce(col("online_order_count"), lit(0)).alias("online_orders"),
            coalesce(col("offline_order_count"), lit(0)).alias("offline_orders"),
            col("first_order_date"),
            col("last_order_date"),
            datediff(current_date(), col("last_order_date")).alias("days_since_last_order"),
            datediff(col("last_order_date"), col("first_order_date")).alias("customer_tenure_days")
        )
        .withColumn("online_order_pct",
            when(col("lifetime_orders") > 0,
                col("online_orders") / col("lifetime_orders") * 100
            ).otherwise(0)
        )
        .withColumn("avg_units_per_order",
            when(col("lifetime_orders") > 0,
                col("total_units") / col("lifetime_orders")
            ).otherwise(0)
        )
        .withColumn("customer_status",
            when(col("lifetime_orders").isNull() | (col("lifetime_orders") == 0), "Inactive")
            .when(col("days_since_last_order") <= 90, "Active")
            .when(col("days_since_last_order") <= 365, "At Risk")
            .otherwise("Churned")
        )
    )
    
    # Write to Gold
    (gold_customer_360
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{GOLD_SCHEMA}.gold_customer_360")
    )
    
    duration = time.time() - start_time
    row_count = gold_customer_360.count()
    
    transformation_results.append({
        "table": "gold_customer_360",
        "status": "success",
        "row_count": row_count,
        "duration_seconds": round(duration, 2)
    })
    
    print(f"\n✅ gold_customer_360 created: {row_count:,} rows in {duration:.2f}s")
    
except Exception as e:
    print(f"\n❌ Failed: {str(e)}")
    transformation_results.append({"table": "gold_customer_360", "status": "failed", "row_count": 0, "duration_seconds": round(time.time() - start_time, 2)})

In [0]:
# Customer Segmentation using RFM Analysis
print("\n" + "=" * 80)
print("🎯 BUILDING gold_customer_segmentation")
print("=" * 80)

start_time = time.time()

try:
    # Read from Silver
    fact_sales_order = spark.table(f"{SILVER_SCHEMA}.fact_sales_order")
    dim_customer = spark.table(f"{SILVER_SCHEMA}.dim_customer").filter(col("is_current") == True)
    dim_date = spark.table(f"{SILVER_SCHEMA}.dim_date")
    
    # Calculate RFM metrics
    reference_date = current_date()
    
    rfm_data = (fact_sales_order
        .join(dim_date, fact_sales_order.order_date_sk == dim_date.date_sk)
        .groupBy(col("customer_sk"))
        .agg(
            max("date_value").alias("last_purchase_date"),
            count("sales_order_id").alias("frequency"),
            sum("total_due_amount").alias("monetary")
        )
        .withColumn("recency", datediff(lit(reference_date), col("last_purchase_date")))
    )
    
    # Calculate RFM scores (1-5 scale)
    rfm_scored = (rfm_data
        .withColumn(
            "recency_score",
            ntile(5).over(Window.orderBy(col("recency").asc()))
        )
        .withColumn(
            "frequency_score",
            ntile(5).over(Window.orderBy(col("frequency").desc()))
        )
        .withColumn(
            "monetary_score",
            ntile(5).over(Window.orderBy(col("monetary").desc()))
        )
        .withColumn("rfm_score", col("recency_score") + col("frequency_score") + col("monetary_score"))
    )
    
    # Assign segments based on RFM scores
    gold_customer_segmentation = (rfm_scored
        .join(dim_customer, rfm_scored.customer_sk == dim_customer.dim_customer_sk)
        .select(
            col("customer_id"),
            col("first_name"),
            col("last_name"),
            col("email_address"),
            col("recency"),
            col("frequency"),
            col("monetary"),
            col("recency_score"),
            col("frequency_score"),
            col("monetary_score"),
            col("rfm_score")
        )
        .withColumn("customer_segment",
            when((col("recency_score") >= 4) & (col("frequency_score") >= 4) & (col("monetary_score") >= 4), "Champions")
            .when((col("recency_score") >= 3) & (col("frequency_score") >= 3) & (col("monetary_score") >= 3), "Loyal Customers")
            .when((col("recency_score") >= 4) & (col("frequency_score") <= 2), "New Customers")
            .when((col("recency_score") <= 2) & (col("frequency_score") >= 4), "At Risk")
            .when((col("recency_score") <= 2) & (col("frequency_score") <= 2) & (col("monetary_score") >= 4), "Cant Lose Them")
            .when((col("recency_score") <= 2) & (col("frequency_score") <= 2) & (col("monetary_score") <= 2), "Hibernating")
            .when((col("recency_score") >= 3) & (col("frequency_score") <= 2) & (col("monetary_score") <= 2), "Promising")
            .otherwise("Needs Attention")
        )
        .withColumn("segment_priority",
            when(col("customer_segment") == "Champions", 1)
            .when(col("customer_segment") == "Loyal Customers", 2)
            .when(col("customer_segment") == "Cant Lose Them", 3)
            .when(col("customer_segment") == "At Risk", 4)
            .when(col("customer_segment") == "Promising", 5)
            .when(col("customer_segment") == "New Customers", 6)
            .when(col("customer_segment") == "Needs Attention", 7)
            .otherwise(8)
        )
    )
    
    # Write to Gold
    (gold_customer_segmentation
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{GOLD_SCHEMA}.gold_customer_segmentation")
    )
    
    duration = time.time() - start_time
    row_count = gold_customer_segmentation.count()
    
    transformation_results.append({
        "table": "gold_customer_segmentation",
        "status": "success",
        "row_count": row_count,
        "duration_seconds": round(duration, 2)
    })
    
    print(f"\n✅ gold_customer_segmentation created: {row_count:,} rows in {duration:.2f}s")
    
    # Show segment distribution
    segment_dist = gold_customer_segmentation.groupBy("customer_segment").count().orderBy("count", ascending=False)
    print("\n📊 Customer Segment Distribution:")
    display(segment_dist)
    
except Exception as e:
    print(f"\n❌ Failed: {str(e)}")
    transformation_results.append({"table": "gold_customer_segmentation", "status": "failed", "row_count": 0, "duration_seconds": round(time.time() - start_time, 2)})

In [0]:
# Customer Lifetime Value Analysis
print("\n" + "=" * 80)
print("💵 BUILDING gold_customer_lifetime_value")
print("=" * 80)

start_time = time.time()

try:
    # Read gold customer 360
    gold_customer_360 = spark.table(f"{GOLD_SCHEMA}.gold_customer_360")
    
    # Calculate CLV metrics
    gold_clv = (gold_customer_360
        .withColumn("annual_revenue",
            when(col("customer_tenure_days") > 0,
                col("lifetime_revenue") / col("customer_tenure_days") * 365
            ).otherwise(col("lifetime_revenue"))
        )
        .withColumn("estimated_clv_3year",
            col("annual_revenue") * 3 * 0.85  # 3 years with 15% discount for churn
        )
        .withColumn("clv_tier",
            when(col("estimated_clv_3year") >= 10000, "Platinum")
            .when(col("estimated_clv_3year") >= 5000, "Gold")
            .when(col("estimated_clv_3year") >= 2000, "Silver")
            .otherwise("Bronze")
        )
        .select(
            col("customer_id"),
            col("full_name"),
            col("email_address"),
            col("company_name"),
            col("lifetime_orders"),
            col("lifetime_revenue"),
            col("avg_order_value"),
            col("customer_tenure_days"),
            col("days_since_last_order"),
            col("customer_status"),
            col("annual_revenue"),
            col("estimated_clv_3year"),
            col("clv_tier"),
            col("shipping_country"),
            col("shipping_state")
        )
    )
    
    # Add value ranking
    gold_clv = gold_clv.withColumn(
        "clv_rank",
        dense_rank().over(Window.orderBy(col("estimated_clv_3year").desc()))
    )
    
    # Write to Gold
    (gold_clv
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{GOLD_SCHEMA}.gold_customer_lifetime_value")
    )
    
    duration = time.time() - start_time
    row_count = gold_clv.count()
    
    transformation_results.append({
        "table": "gold_customer_lifetime_value",
        "status": "success",
        "row_count": row_count,
        "duration_seconds": round(duration, 2)
    })
    
    print(f"\n✅ gold_customer_lifetime_value created: {row_count:,} rows in {duration:.2f}s")
    
    # Show CLV tier distribution
    clv_dist = gold_clv.groupBy("clv_tier").agg(
        count("*").alias("customer_count"),
        sum("estimated_clv_3year").alias("total_clv")
    ).orderBy("total_clv", ascending=False)
    print("\n💰 CLV Tier Distribution:")
    display(clv_dist)
    
except Exception as e:
    print(f"\n❌ Failed: {str(e)}")
    transformation_results.append({"table": "gold_customer_lifetime_value", "status": "failed", "row_count": 0, "duration_seconds": round(time.time() - start_time, 2)})

In [0]:
# Summary
print("\n" + "=" * 80)
print("📊 CUSTOMER 360 ANALYTICS SUMMARY")
print("=" * 80)

success_count = sum(1 for r in transformation_results if r["status"] == "success")
failed_count = sum(1 for r in transformation_results if r["status"] == "failed")
total_rows = sum(r["row_count"] for r in transformation_results)

print(f"\nGold Tables Created: {len(transformation_results)}")
print(f"  ✅ Success: {success_count}")
print(f"  ❌ Failed: {failed_count}")
print(f"Total Rows Created: {total_rows:,}")

summary_df = spark.createDataFrame([Row(**r) for r in transformation_results])
print("\nDetailed Results:")
display(summary_df.orderBy("status", "table"))

print("\n" + "=" * 80)
if failed_count == 0:
    print("✅ GOLD CUSTOMER 360 COMPLETED SUCCESSFULLY")
    print("=" * 80)
    print(f"Completion Time: {datetime.now()}")
    dbutils.notebook.exit("SUCCESS")
else:
    print("⚠️ GOLD CUSTOMER 360 COMPLETED WITH ERRORS")
    print("=" * 80)
    print(f"Completion Time: {datetime.now()}")
    dbutils.notebook.exit(f"PARTIAL_SUCCESS: {failed_count} tables failed")